In [1]:
!pip install openneuro-py mne mne-bids

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.7/890.7 kB 6.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [mne-bids]7/9 [openneuro-py]e]


In [ ]:
"""Preprocess the Nencki-Symfonia oddball EEG recording.

The notebook version of this workflow is exploratory.  This script keeps the
same processing steps while making paths explicit and grouping related steps
into reusable functions.

Examples
--------
Run the preprocessing using the data already present in ``data/``::

    python preprocess.py

Download the selected OpenNeuro files before processing::

    python preprocess.py --download

Open the diagnostic MNE plots at the end of the run::

    python preprocess.py --plot
"""

from __future__ import annotations

import argparse
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import mne
import numpy as np
import openneuro
import pandas as pd
from mne.channels import get_builtin_montages
from mne_bids import BIDSPath, read_raw_bids


# PROJECT_ROOT = Path(__file__).resolve().parent
PROJECT_ROOT_KAGGLE = Path("/kaggle") / "input" / "datasets"
DATASET_NAME = Path("trungdttik55") / "nencki-symfonia-eeg-oddball"
DEFAULT_DATA_ROOT = PROJECT_ROOT_KAGGLE / DATASET_NAME / "data"
SUBJECT = "01"
TASK = "oddball"

STIMULUS_EVENT_ID = {
    "standard": 5,
    "distractor": 7,
    "target": 6,
}
RESPONSE_EVENT_ID = {"response": 1}
MONTAGE_NAME = "colin27_1005"
ICA_EXCLUDE = [0, 11, 21, 26]

def normalize_subject(subject: int | str) -> str:
    """Validate a subject number and return its two-digit BIDS identifier."""

    try:
        subject_number = int(subject)
    except (TypeError, ValueError) as error:
        raise ValueError("subject must be an integer from 1 to 42") from error

    if not 1 <= subject_number <= 42:
        raise ValueError("subject must be an integer from 1 to 42")
    return f"{subject_number:02d}"

def load_raw(
    data_root: Path,
    subject: int | str = SUBJECT,
) -> mne.io.BaseRaw:
    """Load the selected subject's oddball BrainVision recording."""

    subject_id = normalize_subject(subject)

    bids_path = BIDSPath(
        subject=subject_id,
        task=TASK,
        datatype="eeg",
        root=str(data_root),
    )
    raw = read_raw_bids(bids_path=bids_path, verbose=False, on_ch_mismatch='warn')
    raw.load_data()
    raw.pick("eeg")
    return raw


def filter_for_erp(raw: mne.io.BaseRaw) -> mne.io.BaseRaw:
    """Apply the 50 Hz notch and the 0.1-40 Hz ERP band-pass filters."""

    raw.notch_filter(50)
    raw.filter(0.1, 40.0)
    return raw


def build_events_and_epochs(
    raw: mne.io.BaseRaw,
    data_root: Path,
    subject: int | str = SUBJECT,
) -> tuple[np.ndarray, mne.Epochs, mne.Epochs]:
    """Read the selected subject's event TSV and create epochs."""

    subject_id = normalize_subject(subject)

    events_path = (
        data_root
        / f"sub-{subject_id}"
        / "eeg"
        / f"sub-{subject_id}_task-{TASK}_events.tsv"
    )
    events_tsv = pd.read_csv(events_path, sep="\t")

    events_tsv = events_tsv.dropna(subset=['trial_type'])

    # Event labels such as ``S5`` are converted into integer sample codes.

    event_codes = (
        events_tsv["event_type"]
        .str.replace(" ", "", regex=False)
        .str.extract(r"S(\d+)")[0]
        .astype(int)
    )
    events = np.column_stack(
        [
            np.round(events_tsv["onset"].to_numpy() * raw.info["sfreq"]).astype(int),
            np.zeros(len(events_tsv), dtype=int),
            event_codes.to_numpy(),
        ]
    )

    stimulus_events = events[
        np.isin(events[:, 2], list(STIMULUS_EVENT_ID.values()))
    ]
    stimulus_epochs = mne.Epochs(
        raw,
        stimulus_events,
        event_id=STIMULUS_EVENT_ID,
        tmin=-0.2,
        tmax=0.8,
        baseline=(-0.2, 0),
        preload=True,
    )

    response_events = events[events[:, 2] == RESPONSE_EVENT_ID["response"]]
    response_epochs = mne.Epochs(
        raw,
        response_events,
        event_id=RESPONSE_EVENT_ID,
        tmin=-0.2,
        tmax=0.8,
        baseline=(-0.2, 0),
        preload=True,
    )
    return events, stimulus_epochs, response_epochs


def compare_builtin_montages(channel_names: list[str]) -> pd.DataFrame:
    """Find which built-in MNE montage contains the most recording channels."""

    montage_matches: list[dict[str, Any]] = []
    for montage_name in get_builtin_montages():
        candidate = mne.channels.make_standard_montage(montage_name)
        candidate_names = candidate.get_positions()["ch_pos"].keys()
        matched_names = [name for name in channel_names if name in candidate_names]
        montage_matches.append(
            {
                "montage": montage_name,
                "matched": len(matched_names),
                "total": len(channel_names),
                "missing": len(channel_names) - len(matched_names),
            }
        )

    return pd.DataFrame(montage_matches).sort_values(
        "matched", ascending=False
    ).head(15)


def fit_ica(raw: mne.io.BaseRaw) -> tuple[mne.io.BaseRaw, mne.preprocessing.ICA]:
    """Prepare a 1-40 Hz, 250 Hz copy and fit the exploratory ICA model."""

    raw_ica = raw.copy().filter(1.0, 40.0)
    raw_ica.resample(250)

    ica = mne.preprocessing.ICA(
        n_components=30,
        random_state=42,
        max_iter="auto",
    )
    ica.fit(raw_ica)
    return raw_ica, ica


def apply_montage_and_fit_ica(
    raw: mne.io.BaseRaw,
) -> tuple[mne.io.BaseRaw, mne.preprocessing.ICA]:
    """Set the selected montage and fit ICA using only channels with positions."""

    montage = mne.channels.make_standard_montage(MONTAGE_NAME)
    raw.set_montage(montage, on_missing="ignore")

    raw_ica = raw.copy().filter(1.0, 40.0)
    raw_ica.resample(250)
    valid_channels = [
        channel_name
        for channel_name in raw_ica.ch_names
        if channel_name in montage.get_positions()["ch_pos"]
    ]
    raw_ica.pick(valid_channels)

    ica = mne.preprocessing.ICA(
        n_components=30,
        random_state=42,
        max_iter="auto",
    )
    ica.fit(raw_ica)
    return raw_ica, ica


def create_diagnostic_plots(
    raw: mne.io.BaseRaw,
    raw_ica: mne.io.BaseRaw,
    ica: mne.preprocessing.ICA,
) -> None:
    """Reproduce the notebook's montage, ICA, overlay, and raw plots."""

    raw.get_montage().plot()
    ica.plot_components()
    ica.plot_properties(raw_ica)
    ica.plot_overlay(raw_ica, exclude=ICA_EXCLUDE)
    raw_ica.plot()

In [ ]:
def preprocess(
    subject: int | str = SUBJECT,
    data_root: Path = DEFAULT_DATA_ROOT,
    show_plots: bool = False,
 ) -> dict[str, Any]:
    """Preprocess one subject from 1 through 42 and return its results."""

    subject_id = normalize_subject(subject)

    raw = load_raw(data_root, subject_id)
    print(raw.info)

    # The notebook first uses the ERP-filtered signal for event epoching.
    filter_for_erp(raw)
    events_from_annotations, event_id_all = mne.events_from_annotations(raw)
    print("Events found in annotations:", event_id_all)

    # This exploratory fit mirrors the first ICA block in the notebook.
    exploratory_raw_ica, exploratory_ica = fit_ica(raw)

    # Apply the chosen montage and refit ICA on channels with known positions.
    raw_ica, ica = apply_montage_and_fit_ica(raw)
    ica.exclude = ICA_EXCLUDE
    raw.set_eeg_reference("average")

    # The notebook performs a final fit on a copy after setting the montage.
    raw_fitted = raw.copy()
    ica.fit(raw_fitted)
    ica.apply(raw_fitted)

    events, epochs, response_epochs = build_events_and_epochs(
        raw_fitted, data_root, subject_id
    )
    print("Stimulus epochs:", epochs)
    print("Response epochs:", response_epochs)
    print("Raw data shape:", raw_fitted.get_data().shape)
    print("Stimulus epochs shape:", epochs.get_data().shape)

    print(compare_builtin_montages(raw.ch_names))

    if show_plots:
        create_diagnostic_plots(raw, raw_fitted, ica)
        plt.show()

    print("Done processed subject {}".format(str(subject).zfill(2)))

    return {
        "raw": raw,
        "events_from_annotations": events_from_annotations,
        "event_id_all": event_id_all,
        "events": events,
        "epochs": epochs,
        "response_epochs": response_epochs,
        "exploratory_raw_ica": exploratory_raw_ica,
        "exploratory_ica": exploratory_ica,
        "raw_ica": raw_ica,
        "raw_ica_fitted": raw_fitted,
        "ica": ica,
        "subject": subject_id,
    }

In [ ]:
preprocess(1)